<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH33_RAG_Part_II_Chatting_with_Your_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -qU supabase langchain-community
!pip install -qU langchain-groq langchain-text-splitters langchain-core
!pip install -qU langchain-huggingface
!pip install -qU langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.1 MB/s eta 0:00:00


In [4]:
import os
from google.colab import userdata
from supabase.client import create_client, Client
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_huggingface import HuggingFaceEmbeddings # Updated import
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

# ==========================================
# 1단계: 환경 변수 및 인증 정보 설정
# ==========================================
# 1) Supabase 접속 정보 (앞서 확인한 API 주소와 anon 키)
SUPABASE_URL = "https://tllrshlupjcikzaywaqm.supabase.co"
SUPABASE_KEY = userdata.get('supabase')

supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 2) Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

# ==========================================
# 2단계: 데이터 로드, 분할 및 임베딩 (Ingestion)
# ==========================================
# 문서 로드 및 분할
raw_documents = TextLoader('./test.txt', encoding='utf-8').load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

# 수파베이스와 연동할 HuggingFace 384차원 임베딩 모델 설정
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Supabase VectorStore에 데이터 Store (인덱싱 및 저장)
db = SupabaseVectorStore.from_documents(
    documents=documents,
    embedding=embeddings_model,
    client=supabase_client,
    table_name="documents",
    query_name="match_documents"  # SQL Editor에 등록한 함수 이름
)

# ==========================================
# 3단계: 검색기(Retriever) 및 프롬프트, Groq LLM 설정
# ==========================================
# 유사한(cosine similarity) 문서 2개를 뽑아오는 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k": 2})

query = 'Who are the key figures in the ancient greek history of philosophy?'

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}

    Question: {question}"""
)

# 초고속 가성비 모델인 Groq(Llama 3) 모델로 교체
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)


# ==========================================
# 4단계: LCEL @chain 데코레이터를 활용한 효율적인 파이썬 함수 캡슐화 (Generation)
# ==========================================
print("Running RAG system using Groq and Supabase...\n")

@chain
def qa_chain(input_query):
    # 1. Retrieval: 질문과 관련된 문서 조각 가져오기
    docs = retriever.invoke(input_query)

    # 2. Prompt 조립: 질문과 컨텍스트 결합
    formatted = prompt.invoke({"context": docs, "question": input_query})

    # 3. Generation: Groq LLM을 통해 답변 생성
    answer = llm.invoke(formatted)
    return answer

# RAG 체인 실행 및 결과 출력
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Running RAG system using Groq and Supabase...

[최종 답변 결과]
Unfortunately, the provided context does not mention specific key figures in ancient Greek philosophy. It only mentions that "Greek philosophers and scientists" laid the groundwork for diverse fields of study, but it does not provide any names or details about individual philosophers.


# **Query Contruction**

In [ ]:
# @title **Text-to-Metadata Filter**
# Most vector stores provide the ability to limit your vector search based on metadata.
# During the embedding process, we can attach metadata key-value pairs to vectors
# in an index and then later specify filter expressions when you query the index.

In [24]:
!pip install -q -U langchain-classic langchain-groq lark

In [28]:
from langchain_classic.chains.query_constructor.base import load_query_constructor_runnable
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_groq import ChatGroq

# 1. 메타데이터 필드 정의
fields = [
    AttributeInfo(name="genre", description="영화 장르 (SF, 로맨스, 스릴러 등)", type="string"),
    AttributeInfo(name="year", description="영화 개봉 연도", type="integer"),
    AttributeInfo(name="rating", description="1~10 사이 평점", type="float"),
]

# 2. LLM 준비(llama-3.1-8b-instant를 llama-3.3-70b-versatile로 Upgrade)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# 3. 자연어 → 구조화된 쿼리로 변환하는 체인
query_constructor = load_query_constructor_runnable(
    llm,
    document_contents="영화 줄거리 요약",
    attribute_info=fields,
)

# 4. 실행
result = query_constructor.invoke({"query": "평점 8.5 이상인 SF 영화 알려줘"})
print(result)

query='SF' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='rating', value=8.5), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='genre', value='SF')]) limit=None


In [60]:
# @title **Text-to-SQL**

from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langchain_groq import ChatGroq
import re

# Chinook.db 다운받기
!wget https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite -O Chinook.db


db = SQLDatabase.from_uri("sqlite:///Chinook.db")
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

write_query = create_sql_query_chain(llm, db)
execute_query = QuerySQLDatabaseTool(db=db)

def extract_sql(text: str) -> str:
    # 마크다운 코드블록 안에 SQL이 있으면 그 부분만 추출
    match = re.search(r"```sql\s*(.*?)\s*```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # 코드블록이 없으면 SELECT/INSERT/UPDATE/DELETE로 시작하는 줄부터 끝까지 추출
    match = re.search(r"(SELECT|INSERT|UPDATE|DELETE|WITH)\b.*", text, re.IGNORECASE | re.DOTALL)
    return match.group(0).strip() if match else text.strip()

chain = write_query | extract_sql | execute_query

result = chain.invoke({"question": "How many employees are there?"})
print(result)

--2026-07-20 06:45:32--  https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1007616 (984K) [application/octet-stream]
Saving to: ‘Chinook.db’

Chinook.db          100%[===================>] 984.00K  --.-KB/s    in 0.1s    

2026-07-20 06:45:32 (8.17 MB/s) - ‘Chinook.db’ saved [1007616/1007616]

[(8,)]


In [58]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [63]:
import pandas as pd
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('Chinook.db')

# Read the 'Employee' table into a pandas DataFrame
employees_df = pd.read_sql_query("SELECT * FROM Employee", conn)

# Close the connection
conn.close()

employees_df

,EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
0,1,Adams,Andrew,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,Edwards,Nancy,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,Peacock,Jane,Sales Support Agent,2.0,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,Park,Margaret,Sales Support Agent,2.0,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,Johnson,Steve,Sales Support Agent,2.0,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,Mitchell,Michael,IT Manager,1.0,1973-07-01 00:00:00,2003-10-17 00:00:00,5827 Bowness Road NW,Calgary,AB,Canada,T3B 0C5,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,King,Robert,IT Staff,6.0,1970-05-29 00:00:00,2004-01-02 00:00:00,590 Columbia Boulevard West,Lethbridge,AB,Canada,T1K 5N8,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,Callahan,Laura,IT Staff,6.0,1968-01-09 00:00:00,2004-03-04 00:00:00,923 7 ST NW,Lethbridge,AB,Canada,T1H 1Y8,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com
